# Tiny GPT: Tokenizer First, Then Transformer

This notebook now follows the full beginner pipeline in the right order:

1. install the required libraries
2. download TinyStories from Hugging Face
3. train a custom byte-level BPE tokenizer
4. tokenize the corpus into integer ids
5. build a small GPT-style Transformer
6. train the model on next-token prediction
7. generate text

The design stays simple on purpose so you can see the complete workflow in one place.

In [ ]:
# Run this cell first in Colab to install the required notebook packages.

import importlib
import subprocess
import sys


def ensure_package(module_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(module_name)
        print(f"{module_name} is already installed.")
    except ImportError:
        package_name = pip_name or module_name
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


ensure_package("torch")
ensure_package("datasets")
ensure_package("regex")
ensure_package("tqdm")

In [ ]:
import math
from collections import Counter
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    import regex as re
except ImportError:
    import re

try:
    from datasets import load_dataset
except ImportError:
    load_dataset = None

torch.manual_seed(42)


def resolve_device() -> tuple[torch.device, str]:
    """Pick the best available training device in the order CUDA, MPS, CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda"), "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps"), "mps"
    return torch.device("cpu"), "cpu"


device, device_name = resolve_device()
print(torch.__version__)
print("device:", device)
print("device type:", device_name)

## Step 1: Load TinyStories

We train the tokenizer on **TinyStories** from Hugging Face.

To keep the notebook beginner-friendly and fast, we only use a small slice of the training split. You can increase the number of stories later if you want a better tokenizer.

In [ ]:
DATASET_NAME = "roneneldan/TinyStories"
MAX_STORIES = 500
END_OF_TEXT = "<|endoftext|>"


def load_tinystories_text(max_stories: int) -> tuple[str, list[str], str]:
    """Load TinyStories text, or fall back to a small local sample if offline."""
    if load_dataset is None:
        source = "offline fallback sample"
        stories = [
            "Once upon a time there was a small blue bird who loved to sing.",
            "A child found a shiny key under a tree and wondered what it could open.",
            "The moon looked bright over the quiet town while the wind moved softly.",
        ] * 200
        joined_text = END_OF_TEXT.join(stories)
        return joined_text, stories, source

    try:
        dataset = load_dataset(DATASET_NAME, split=f"train[:{max_stories}]")
        stories = [row["text"] for row in dataset if row["text"].strip()]
        source = f"{DATASET_NAME} train[:{max_stories}]"
    except Exception as exc:
        print("TinyStories download failed, using fallback sample instead.")
        print(type(exc).__name__, exc)
        source = "offline fallback sample"
        stories = [
            "Once upon a time there was a small blue bird who loved to sing.",
            "A child found a shiny key under a tree and wondered what it could open.",
            "The moon looked bright over the quiet town while the wind moved softly.",
        ] * 200

    joined_text = END_OF_TEXT.join(stories)
    return joined_text, stories, source


raw_text, stories, data_source = load_tinystories_text(MAX_STORIES)
print("data source:", data_source)
print("stories:", len(stories))
print("characters:", len(raw_text))
print(raw_text[:400])

## Step 2: Train a Custom Tokenizer

This notebook uses a simple **byte-level BPE tokenizer**.

Important ideas:
- every text string becomes UTF-8 bytes
- the base vocabulary starts with 256 single-byte tokens
- we repeatedly merge the most common adjacent byte pair
- the learned merge list becomes the tokenizer

This is a beginner version of the same general idea used in the project tokenizer code.

In [ ]:
PRETOKEN_PATTERN = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+")


def pretoken_to_bytes(pretoken: str) -> tuple[bytes, ...]:
    """Convert one pretoken into a tuple of one-byte tokens."""
    return tuple(bytes([b]) for b in pretoken.encode("utf-8"))


def iter_pairs(tokens: tuple[bytes, ...]):
    """Yield each adjacent pair in a token sequence."""
    return zip(tokens, tokens[1:])


def merge_pair_in_sequence(
    tokens: tuple[bytes, ...],
    pair: tuple[bytes, bytes],
    merged_token: bytes,
) -> tuple[bytes, ...]:
    """Merge one pair left to right inside a token sequence."""
    merged = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
            merged.append(merged_token)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return tuple(merged)


def split_with_special_tokens(text: str, special_tokens: list[str]) -> list[tuple[bool, str]]:
    """Split text into ordinary chunks and exact special-token matches."""
    if not special_tokens:
        return [(False, text)]

    sorted_tokens = sorted(special_tokens, key=len, reverse=True)
    pattern = re.compile("|".join(re.escape(token) for token in sorted_tokens))
    pieces = []
    cursor = 0

    for match in pattern.finditer(text):
        if match.start() > cursor:
            pieces.append((False, text[cursor:match.start()]))
        pieces.append((True, match.group(0)))
        cursor = match.end()

    if cursor < len(text):
        pieces.append((False, text[cursor:]))

    return pieces


class SimpleBPETokenizer:
    """A small byte-level BPE tokenizer for learning and experiments."""

    def __init__(
        self,
        vocab: dict[int, bytes],
        merges: list[tuple[bytes, bytes]],
        special_tokens: list[str] | None = None,
    ) -> None:
        self.vocab = dict(vocab)
        self.merges = list(merges)
        self.special_tokens = list(special_tokens or [])

        self.merge_ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.id_to_bytes = dict(self.vocab)
        self.bytes_to_id = {token_bytes: token_id for token_id, token_bytes in self.id_to_bytes.items()}
        self._encode_cache = {}

    @classmethod
    def train(
        cls,
        text: str,
        vocab_size: int,
        special_tokens: list[str] | None = None,
    ) -> "SimpleBPETokenizer":
        """Train a small byte-level BPE tokenizer directly from text."""
        special_tokens = list(special_tokens or [])
        base_vocab_size = 256 + len(special_tokens)
        if vocab_size < base_vocab_size:
            raise ValueError(
                f"vocab_size must be at least {base_vocab_size} to include special tokens and bytes."
            )

        sequence_counts = Counter()
        for is_special, chunk in split_with_special_tokens(text, special_tokens):
            if is_special:
                continue
            for match in PRETOKEN_PATTERN.finditer(chunk):
                pretoken = match.group(0)
                if pretoken:
                    sequence_counts[pretoken_to_bytes(pretoken)] += 1

        vocab = {}
        next_id = 0
        for token in special_tokens:
            vocab[next_id] = token.encode("utf-8")
            next_id += 1

        for byte_value in range(256):
            vocab[next_id] = bytes([byte_value])
            next_id += 1

        vocab_values = set(vocab.values())
        merges = []

        while len(vocab) < vocab_size:
            pair_counts = Counter()
            for tokens, count in sequence_counts.items():
                for pair in iter_pairs(tokens):
                    pair_counts[pair] += count

            if not pair_counts:
                break

            best_pair = max(pair_counts.items(), key=lambda item: (item[1], item[0]))[0]
            merged_token = best_pair[0] + best_pair[1]
            merges.append(best_pair)

            if merged_token not in vocab_values:
                vocab[next_id] = merged_token
                vocab_values.add(merged_token)
                next_id += 1

            updated_counts = Counter()
            for tokens, count in sequence_counts.items():
                updated_tokens = merge_pair_in_sequence(tokens, best_pair, merged_token)
                updated_counts[updated_tokens] += count
            sequence_counts = updated_counts

        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)

    def _apply_merges(self, pretoken: str) -> list[bytes]:
        """Apply learned merges greedily by merge order."""
        tokens = list(pretoken_to_bytes(pretoken))
        while len(tokens) >= 2:
            best_rank = None
            best_pair = None

            for pair in iter_pairs(tuple(tokens)):
                rank = self.merge_ranks.get(pair)
                if rank is None:
                    continue
                if best_rank is None or rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            merged_token = best_pair[0] + best_pair[1]
            tokens = list(merge_pair_in_sequence(tuple(tokens), best_pair, merged_token))

        return tokens

    def encode(self, text: str) -> list[int]:
        """Encode text into tokenizer ids."""
        ids = []
        for is_special, chunk in split_with_special_tokens(text, self.special_tokens):
            if not chunk:
                continue
            if is_special:
                ids.append(self.bytes_to_id[chunk.encode("utf-8")])
                continue

            for match in PRETOKEN_PATTERN.finditer(chunk):
                pretoken = match.group(0)
                if pretoken in self._encode_cache:
                    ids.extend(self._encode_cache[pretoken])
                    continue

                merged_tokens = self._apply_merges(pretoken)
                token_ids = [self.bytes_to_id[token] for token in merged_tokens]
                self._encode_cache[pretoken] = token_ids
                ids.extend(token_ids)

        return ids

    def decode(self, ids: list[int]) -> str:
        """Decode tokenizer ids back into text."""
        raw_bytes = b"".join(self.id_to_bytes[token_id] for token_id in ids)
        return raw_bytes.decode("utf-8", errors="replace")

In [ ]:
TOKENIZER_VOCAB_SIZE = 384
SPECIAL_TOKENS = [END_OF_TEXT]

tokenizer = SimpleBPETokenizer.train(
    text=raw_text,
    vocab_size=TOKENIZER_VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS,
)

print("tokenizer vocab size:", len(tokenizer.vocab))
print("number of learned merges:", len(tokenizer.merges))

sample_text = stories[0][:200]
sample_ids = tokenizer.encode(sample_text)

print("sample text:")
print(sample_text)
print()
print("sample token ids:")
print(sample_ids[:80])
print()
print("decoded sample:")
print(tokenizer.decode(sample_ids))

## Step 3: Tokenize the Corpus

Now that the tokenizer is trained, we convert the full TinyStories text into integer token ids.

These ids are what the GPT model will see during training.

In [ ]:
all_token_ids = tokenizer.encode(raw_text)
token_data = torch.tensor(all_token_ids, dtype=torch.long)

split_index = int(0.9 * len(token_data))
train_data = token_data[:split_index]
val_data = token_data[split_index:]

print("total tokens:", len(token_data))
print("train tokens:", len(train_data))
print("val tokens:", len(val_data))
print("average characters per token:", round(len(raw_text) / len(token_data), 3))

## Step 4: Build Tiny GPT

Now that we already have token ids, we can build the GPT model.

This notebook keeps the Transformer simple:
- `LayerNorm`
- learned position embeddings
- causal multi-head self-attention
- a basic `GELU` feed-forward network

The project code contains more advanced versions, but the overall structure is the same.

In [ ]:
@dataclass
class TinyGPTConfig:
    """Simple configuration container for the model."""

    vocab_size: int
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 128
    dropout: float = 0.1

    def __post_init__(self):
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head.")


class CausalSelfAttention(nn.Module):
    """Masked multi-head self-attention for a decoder-only Transformer."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer(
            "causal_mask",
            mask.view(1, 1, config.block_size, config.block_size),
            persistent=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, channels = x.shape

        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        attention_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attention_scores = attention_scores.masked_fill(
            self.causal_mask[:, :, :seq_len, :seq_len] == 0,
            float("-inf"),
        )

        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)

        y = attention_weights @ v
        y = y.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)
        y = self.c_proj(y)
        return self.resid_dropout(y)


class FeedForward(nn.Module):
    """Simple MLP used after attention in each Transformer block."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    """One full Transformer block: norm, attention, norm, and MLP."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = FeedForward(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class TinyGPT(nn.Module):
    """A small GPT-style language model built from the blocks above."""

    def __init__(self, config: TinyGPTConfig):
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        batch_size, seq_len = idx.shape
        if seq_len > self.config.block_size:
            raise ValueError(
                f"Sequence length {seq_len} is bigger than block size {self.config.block_size}."
            )

        positions = torch.arange(seq_len, device=idx.device)
        token_embeddings = self.token_embedding(idx)
        position_embeddings = self.position_embedding(positions)

        x = self.dropout(token_embeddings + position_embeddings)
        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(batch_size * seq_len, self.config.vocab_size),
                targets.reshape(batch_size * seq_len),
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

## Step 5: Training Helpers

These helper functions follow the same language-model training idea used in the project:
- sample random token windows
- predict the next token
- evaluate train and validation loss over several batches

In [ ]:
def get_batch(
    source: torch.Tensor,
    batch_size: int,
    block_size: int,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample random token windows and their next-token targets."""
    starts = torch.randint(0, len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss(
    model: TinyGPT,
    train_data: torch.Tensor,
    val_data: torch.Tensor,
    eval_iters: int,
    batch_size: int,
) -> dict[str, float]:
    """Average train and validation loss over a few random batches."""
    losses = {}
    model.eval()
    for split_name, split_data in [("train", train_data), ("val", val_data)]:
        split_losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split_data, batch_size, model.config.block_size, device)
            _, loss = model(x, y)
            split_losses.append(float(loss.detach()))
        losses[split_name] = sum(split_losses) / len(split_losses)
    model.train()
    return losses

## Step 6: Train Tiny GPT

Now we train GPT on the tokenized TinyStories data.

For a notebook demo, we use a very small model and a short run. The goal is to show the full pipeline, not to train a strong language model.

In [ ]:
train_config = TinyGPTConfig(
    vocab_size=len(tokenizer.vocab),
    block_size=64,
    n_layer=2,
    n_head=4,
    n_embd=128,
    dropout=0.1,
)

model = TinyGPT(train_config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

batch_size = 16
max_iters = 60
eval_interval = 20
eval_iters = 5
grad_clip = 1.0

for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model, train_data, val_data, eval_iters, batch_size)
        print(
            f"step {step:03d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f}"
        )

    x, y = get_batch(train_data, batch_size, train_config.block_size, device)
    _, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

print("final training loss:", float(loss.detach()))

## Step 7: Generate Text

We encode a prompt with the custom tokenizer, generate more token ids with GPT, and decode those ids back into text.

In [ ]:
prompt = "Once upon a time"
prompt_ids = tokenizer.encode(prompt)
context = torch.tensor([prompt_ids], dtype=torch.long, device=device)

generated_ids = model.generate(context, max_new_tokens=80)
generated_text = tokenizer.decode(generated_ids[0].tolist())
print(generated_text)

## How This Connects to the Project

This notebook now follows the complete learning order:
- dataset
- tokenizer
- token ids
- GPT model
- training
- generation

It stays simpler than the full project code, but the main ideas line up with:
- `bpe_tokenizer/trainer.py`
- `bpe_tokenizer/tokenizer.py`
- `GPT/transformer_lm.py`
- `GPT/transformer_block.py`
- `GPT/multihead_attention.py`
- `GPT/training.py`

Main simplifications in this notebook:
- a smaller BPE trainer written directly in the notebook
- a smaller TinyStories slice
- `LayerNorm` instead of `RMSNorm`
- learned position embeddings instead of `RoPE`
- `GELU` instead of `SwiGLU`
- a short training run for demonstration